## Nome: Caio Semblano Silva
## Matrícula: 621

---

# Atividade 6 - Interpolação de Lagrange

### Função de interpolação de Lagrange

In [5]:
import numpy as np
import matplotlib.pyplot as plt

In [6]:
def lagrange_base(xs, k, x):
    # xs: nós de interpolação (ex.: valores de irradiação conhecidos da tabela)
    # k : índice do nó atual (para construir L_k)
    # x : ponto ou pontos onde vamos avaliar L_k(x)

    xs = np.asarray(xs, dtype=float)  # garante que xs é array NumPy
    x = np.asarray(x, dtype=float)    # garante que x também pode ser array
    xk = xs[k]                        # valor do nó correspondente ao índice k
    L = np.ones_like(x, dtype=float)  # inicializa L(x) = 1 (vai acumular o produto)

    # Produto: L_k(x) = ∏ (x - xj) / (xk - xj), para j ≠ k
    for j, xj in enumerate(xs):
        if j == k:
            continue                  # não usamos o termo quando j = k
        L *= (x - xj) / (xk - xj)     # multiplica cada fator do produto

    return L                          # retorna o valor de L_k(x)


# Utilizar essa função que calcula o polinômio interpolador de Lagrange P_n(x)
def lagrange_interp(xs, ys, x):
    # xs: nós de interpolação (ex.: valores de irradiação conhecidos)
    # ys: valores da função nos nós (ex.: energias geradas conhecidas)
    # x : ponto(s) onde queremos calcular P_n(x)

    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    x = np.asarray(x, dtype=float)
    P = np.zeros_like(x, dtype=float)  # inicializa P(x) = 0

    # Soma: P_n(x) = Σ yk * L_k(x)
    for k in range(len(xs)):
        P += ys[k] * lagrange_base(xs, k, x)  # acumula cada termo da soma

    return P  # retorna o valor do polinômio em x

### Função para plotar os pontos e o polinômio interpolador

In [7]:


def plot_interpolacao(xs, ys, pontos_estimativa=None):
    """
    Plota o polinômio interpolador de Lagrange a partir dos dados fornecidos.

    Parâmetros:
    -----------
    xs : array-like
        Nós de interpolação (ex.: valores de tempo conhecidos).
    ys : array-like
        Valores da função nos nós (ex.: temperaturas medidas correspondentes).
    pontos_estimativa : lista ou array, opcional
        Pontos onde se deseja destacar as estimativas no gráfico (ex.: [10, 11]).
    """

    # Cria um eixo denso para desenhar o polinômio
    xx = np.linspace(min(xs), max(xs), 600)
    Pn = lagrange_interp(xs, ys, xx)

    plt.figure(figsize=(8,5))

    # Plota os pontos originais da tabela
    plt.scatter(xs, ys, label='Dados (tabela)', color='blue')

    # Plota o polinômio interpolador
    plt.plot(xx, Pn, linestyle='--', label='Polinômio interpolador P_n(x)', color='orange')

    # Se pontos de estimativa forem passados, calcula e marca no gráfico
    if pontos_estimativa is not None:
        pontos_estimativa = np.array(pontos_estimativa)
        valores_estimados = lagrange_interp(xs, ys, pontos_estimativa)
        plt.scatter(pontos_estimativa, valores_estimados, marker='x', s=80, color='red', label='Estimativas')

    # Rótulos e título
    plt.xlabel('Tempo (min)')
    plt.ylabel('Temperatura (°C)')
    plt.title('Interpolação de Lagrange — Tempo vs Temperatura')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

### Parte Prática

Uma equipe de engenharia mecânica está caracterizando a curva de resfriamento de uma peça metálica recém-forjada, essencial para definir o processo de tratamento térmico. Sensores de temperatura foram posicionados na peça e registraram a temperatura em diferentes instantes de tempo após a saída do forno.

Os resultados das medições estão na tabela abaixo:

| Tempo (min) | Temperatura (°C) |
|--------------------|---------------|
| 0                 | 850         |
| 2                 | 620          |
| 4                 | 470          |
| 6                 | 360          |
| 8                 | 290          |
| 10                | 245          |

Com base nesses dados, responda:

1. Faça uma estimativa da temperatura da peça considerando os instantes de 1 min, 5 min, 7 min e 12 min. (Arredondar os resultados para duas casas decimais)

In [8]:
xs = np.array([0, 2, 4, 6, 8, 10], dtype=float)
ys = np.array([850, 620, 470, 360, 290, 245], dtype=float)

tempos_estimativa = np.array([1, 5, 7, 12], dtype=float)
temperaturas_estimadas = lagrange_interp(xs, ys, tempos_estimativa)

print(f"{'Tempo (min)':<14} | {'Temperatura estimada (°C)':<26} | Observação")
print("-" * 78)

for t, T in zip(tempos_estimativa, temperaturas_estimadas):
    tipo = "Extrapolação (fora da tabela)" if t > max(xs) or t < min(xs) else "Interpolação"
    print(f"{t:<14.0f} | {T:<26.2f} | {tipo}")

Tempo (min)    | Temperatura estimada (°C)  | Observação
------------------------------------------------------------------------------
1              | 719.43                     | Interpolação
5              | 410.29                     | Interpolação
7              | 319.94                     | Interpolação
12             | 140.00                     | Extrapolação (fora da tabela)


2. Plote um gráfico com os pontos conhecidos e o polinômio interpolador de Lagrange.

In [ ]:
plot_interpolacao(xs, ys, pontos_estimativa=[1, 5, 7])

### Parte Teórica

1. Qual é a diferença entre interpolar um conjunto de dados e extrapolar a partir dele? Por que a extrapolação costuma ser mais arriscada?

**Resposta:**
Interpolar significa estimar o valor da função **dentro** do intervalo coberto pelos dados conhecidos (neste caso, entre 0 min e 10 min). Extrapolar significa estimar **fora** desse intervalo, como no instante de 12 min.

A extrapolação costuma ser mais arriscada porque o polinômio interpolador só é obrigado a passar pelos pontos da tabela. Fora desse intervalo ele não tem restrição experimental e pode crescer ou oscilar de forma irreal, afastando-se do comportamento físico do resfriamento da peça.

2. Se um novo ponto de medição fosse adicionado à tabela (por exemplo, a temperatura em 3 min), o polinômio de Lagrange calculado anteriormente mudaria? Justifique.

**Resposta:**
Sim, o polinômio mudaria. O interpolador de Lagrange é o polinômio de grau no máximo $n$ que passa exatamente pelos $n+1$ pontos dados. Ao incluir um novo instante (por exemplo, 3 min), o conjunto de nós deixa de ser o mesmo: o grau pode aumentar e o polinômio passa a ser o único que também satisfaz o novo ponto $(3, T(3))$. Em geral, o $P_n(x)$ anterior não passaria por essa medição extra, então a fórmula inteira precisa ser reconstruída.

3. No contexto do resfriamento da peça metálica, o que aconteceria com o polinômio interpolador se dois dos pontos medidos tivessem exatamente o mesmo instante de tempo, mas temperaturas diferentes (por exemplo, devido a um erro de leitura do sensor)? Isso seria um problema para a interpolação de Lagrange?

**Resposta:**
Sim, isso seria um problema. Na interpolação de Lagrange cada base $L_k(x)$ contém o quociente $(x - x_j)/(x_k - x_j)$. Se dois instantes coincidirem, algum denominador $x_k - x_j$ vira zero e a construção fica indefinida.

Além disso, um polinômio (e qualquer função) não pode assumir duas temperaturas diferentes no mesmo instante. Na prática isso indicaria erro de medição: seria preciso corrigir ou descartar um dos pontos antes de interpolar.